In [ ]:
import sqlite3
from datetime import datetime

# ==============================================================================
# 1. PERSISTENCIA DE DATOS (SQLite) - INICIALIZACIÓN
# ==============================================================================
def inicializar_base_datos():
    conn = sqlite3.connect('base_datos_startup_grupo_4.db')
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = ON;") # Activación obligatoria de llaves

    # Creación de 3 tablas relacionadas
    cursor.executescript('''
    CREATE TABLE IF NOT EXISTS clientes (
        id_cliente INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT NOT NULL,
        email TEXT UNIQUE,
        categoria TEXT
    );
    CREATE TABLE IF NOT EXISTS productos (
        sku TEXT PRIMARY KEY,
        nombre_repuesto TEXT NOT NULL,
        precio REAL NOT NULL,
        stock INTEGER
    );
    CREATE TABLE IF NOT EXISTS ventas (
        id_venta INTEGER PRIMARY KEY AUTOINCREMENT,
        id_cliente INTEGER,
        sku_producto TEXT,
        cantidad INTEGER NOT NULL,
        total REAL NOT NULL,
        fecha TEXT,
        FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente),
        FOREIGN KEY (sku_producto) REFERENCES productos(sku)
    );
    ''')
    
    # Inserción de 5 registros mínimos por tabla (Regla de Oro)
    try:
        clientes_data = [
            ('Taller Los Andes', 'contacto@andes.com', 'Oro'),
            ('Motos Rápidas Cali', 'ventas@rapidas.com', 'Plata'),
            ('Distribuidora El Eje', 'gerencia@eleje.com', 'Bronce'),
            ('Rodamientos Medellín', 'info@rodamientos.com', 'Oro'),
            ('Serviteca Capital', 'servi@capital.com', 'Plata')
        ]
        cursor.executemany("INSERT OR IGNORE INTO clientes (nombre, email, categoria) VALUES (?, ?, ?)", clientes_data)

        productos_data = [
            ('SKU-001', 'Filtro de Aire Alto Flujo', 45000, 100),
            ('SKU-002', 'Filtro de Aceite Premium', 25000, 150),
            ('SKU-003', 'Kit de Arrastre Reforzado', 180000, 50),
            ('SKU-004', 'Disco de Freno Ventilado', 95000, 30),
            ('SKU-005', 'Pastillas de Freno Cerámicas', 55000, 80)
        ]
        cursor.executemany("INSERT OR IGNORE INTO productos (sku, nombre_repuesto, precio, stock) VALUES (?, ?, ?, ?)", productos_data)

        ventas_data = [
            (1, 'SKU-001', 10, 450000, '2026-04-01'),
            (2, 'SKU-002', 20, 500000, '2026-04-02'),
            (3, 'SKU-003', 5, 900000, '2026-04-03'),
            (4, 'SKU-004', 12, 1140000, '2026-04-04'),
            (5, 'SKU-005', 15, 825000, '2026-04-05')
        ]
        cursor.executemany("INSERT OR IGNORE INTO ventas (id_cliente, sku_producto, cantidad, total, fecha) VALUES (?, ?, ?, ?, ?)", ventas_data)
        conn.commit()
    except sqlite3.Error as e:
        print(f"Error poblando BD: {e}")
    finally:
        conn.close()

# ==============================================================================
# 2. PROGRAMACIÓN ORIENTADA A OBJETOS (3 Padres, 3 Hijos con Encapsulamiento)
# ==============================================================================

# --- JERARQUÍA 1: CLIENTES ---
class Persona: 
    def __init__(self, nombre):
        self._nombre = nombre # Atributo protegido

class ClienteB2B(Persona): 
    def __init__(self, id_cliente, nombre, email):
        super().__init__(nombre)
        self.__id_cliente = id_cliente # Encapsulado
        self.__email = email           # Encapsulado
    
    # Getters y Setters
    def get_id(self): return self.__id_cliente
    def set_email(self, nuevo_email): self.__email = nuevo_email

# --- JERARQUÍA 2: PRODUCTOS ---
class Articulo: 
    def __init__(self, nombre_articulo, precio_base):
        self._nombre_articulo = nombre_articulo
        self._precio_base = precio_base

class RepuestoMoto(Articulo): 
    def __init__(self, sku, nombre_articulo, precio_base, stock):
        super().__init__(nombre_articulo, precio_base)
        self.__sku = sku     # Encapsulado
        self.__stock = stock # Encapsulado

    def get_sku(self): return self.__sku
    def get_precio(self): return self._precio_base

# --- JERARQUÍA 3: VENTAS ---
class Transaccion: 
    def __init__(self, cantidad):
        self._cantidad = cantidad

class OrdenVenta(Transaccion): 
    def __init__(self, id_cliente, sku_producto, cantidad, total):
        super().__init__(cantidad)
        self.__id_cliente = id_cliente
        self.__sku_producto = sku_producto
        self.__total = total
        self.__fecha = datetime.now().strftime("%Y-%m-%d")

    def guardar_en_bd(self):
        try:
            conn = sqlite3.connect('base_datos_startup_grupo_4.db')
            cursor = conn.cursor()
            cursor.execute("INSERT INTO ventas (id_cliente, sku_producto, cantidad, total, fecha) VALUES (?, ?, ?, ?, ?)", 
                           (self.__id_cliente, self.__sku_producto, self._cantidad, self.__total, self.__fecha))
            conn.commit()
            print("✅ [CREATE] Venta registrada con éxito.")
        except sqlite3.Error as e:
            print(f"⚠️ Error BD: {e}")
        finally:
            conn.close()

# ==============================================================================
# 3. INTERFAZ Y LÓGICA CRUD (Menú Interactivo)
# ==============================================================================
def menu_principal():
    inicializar_base_datos()
    
    while True:
        print("\n" + "="*45)
        print("🏍️ KETOZ V2.0 - SISTEMA NACIONAL 🏍️")
        print("="*45)
        print("1. [CREATE] Registrar nueva venta")
        print("2. [READ] Ver reporte cruzado (INNER JOIN)")
        print("3. [UPDATE] Actualizar categoría de cliente")
        print("4. [DELETE] Eliminar registro de venta")
        print("5. Salir")
        
        opcion = input("Seleccione una operación (1-5): ").strip()
        
        # CREATE
        if opcion == "1":
            try:
                id_cliente = int(input("ID del Cliente (1-5): "))
                sku = input("SKU del producto (Ej: SKU-001): ").strip()
                cantidad = int(input("Cantidad a despachar: "))
                
                conn = sqlite3.connect('base_datos_startup_grupo_4.db')
                cursor = conn.cursor()
                cursor.execute("SELECT precio FROM productos WHERE sku = ?", (sku,))
                resultado = cursor.fetchone()
                conn.close()
                
                if resultado:
                    total_calculado = resultado[0] * cantidad
                    nueva_venta = OrdenVenta(id_cliente, sku, cantidad, total_calculado)
                    nueva_venta.guardar_en_bd()
                else:
                    print("❌ Error: SKU no encontrado.")
            except ValueError:
                print("⚠️ Error: Ingrese números válidos.")
        
        # READ (Con JOIN)
        elif opcion == "2":
            try:
                conn = sqlite3.connect('base_datos_startup_grupo_4.db')
                cursor = conn.cursor()
                print("\n--- REPORTE GENERAL DE VENTAS ---")
                cursor.execute('''
                SELECT ventas.id_venta, clientes.nombre, productos.nombre_repuesto, ventas.cantidad, ventas.total 
                FROM ventas 
                INNER JOIN clientes ON ventas.id_cliente = clientes.id_cliente
                INNER JOIN productos ON ventas.sku_producto = productos.sku
                ''')
                for f in cursor.fetchall():
                    print(f"Venta #{f[0]} | Cliente: {f[1]} | Repuesto: {f[2]} | Cant: {f[3]} | Total: ${f[4]:,.2f}")
            except sqlite3.Error as e: print(f"Error: {e}")
            finally: conn.close()
            
        # UPDATE
        elif opcion == "3":
            try:
                id_c = int(input("Ingrese el ID del Cliente a actualizar (1-5): "))
                nueva_cat = input("Nueva categoría (Oro, Plata, Bronce): ").strip()
                conn = sqlite3.connect('base_datos_startup_grupo_4.db')
                cursor = conn.cursor()
                cursor.execute("UPDATE clientes SET categoria = ? WHERE id_cliente = ?", (nueva_cat, id_c))
                conn.commit()
                print(f"✅ [UPDATE] Categoría actualizada a {nueva_cat} para el cliente ID {id_c}.")
            except sqlite3.Error as e: print(f"Error: {e}")
            finally: conn.close()
            
        # DELETE
        elif opcion == "4":
            try:
                id_v = int(input("Ingrese el ID de la venta a cancelar: "))
                conn = sqlite3.connect('base_datos_startup_grupo_4.db')
                cursor = conn.cursor()
                cursor.execute("DELETE FROM ventas WHERE id_venta = ?", (id_v,))
                conn.commit()
                print(f"✅ [DELETE] Venta #{id_v} eliminada del sistema.")
            except sqlite3.Error as e: print(f"Error: {e}")
            finally: conn.close()

        # SALIR
        elif opcion == "5":
            print("Apagando sistema central... ¡Éxitos!")
            break
        else:
            print("Opción inválida.")

# Arrancar sistema
menu_principal()